# Pre-processing a .CHA file for use in CE analysis

In [1]:
import os
import sys
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
data_path = '../data'
chas_path = os.path.join(data_path, 'chas')

In [3]:
keep_columns = ['document_line_no', 'x_turn_id', 'x_speaker', 'x_utterance','overlapping_text', 'corpus', 'conversation_id', 'y_speaker', 'y_utterance', 'y_turn_id']

In [4]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [5]:
all_files = grab_all_files(chas_path)
len(all_files), all_files[0]

(2020, '../data/chas/CABNC/KC7/KC7RE000.cha')

In [6]:
contained_corpora = set([f.split('/')[3] for f in all_files])
contained_corpora = '-'.join(contained_corpora)
outputs_path = os.path.join(data_path, 'server_ready', contained_corpora+'.parquet')
outputs_path

'../data/server_ready/CABNC.parquet'

# Processing all CHA files

Note: the package used here was developed by Prof. Garber. Cite via:

Garber, L. (2019). CHA file python parser. Zenodo. https://doi.org/10.5281/zenodo.3364020

In [7]:
from shared.CHAFile import *

In [8]:
data = []
for f in all_files:
    
    chacha = ChaFile(f)
    meta_data_pieces = f.replace('.cha', '').split('/')

    for line in chacha.getLines():
        line['file'] = f
        line['overlapping_text'] = bool(re.findall(r"(⌋|⌊|⌉|⌈)", line['text']))

        line['corpus'] = meta_data_pieces[3]

        line['conversation_id'] = line['corpus'] + '-' + meta_data_pieces[-1]

        if 'CORAAL' in f:
            line['conversation_id'] += meta_data_pieces[-3]+'-'+meta_data_pieces[-2]

        if ('CABNC' in f) or ('MICASE' in f):
            line['conversation_id'] += '-' + meta_data_pieces[-2]
        
        data += [line.copy()]

    del chacha

data = pd.DataFrame(data)

In [9]:
data.shape

(151542, 14)

In [10]:
data.head()

,document_line_no,utterance_no,speaker,text,bullet,mor,gra,wor,recipient,file,overlapping_text,corpus,conversation_id,com
0,12,1,S1,"+, my god I don't understand a word .","[4665, 7105]","[{'categoria': 'det:poss', 'lexema': 'my', 'ex...",1|2|DET 2|6|SUBJ 3|6|SUBJ 4|6|AUX 5|4|NEG 6|0|...,my 4665_5405 god I 5405_5585 don't 5585_5...,ADULT,../data/chas/MICASE/interviews/int425jg002.cha,False,MICASE,MICASE-int425jg002-interviews,NaN
1,16,2,S2,&=laughs it's shocking looking isn't it ?,"[8065, 11025]","[{'categoria': 'pro:per', 'lexema': 'it', 'ext...",1|3|SUBJ 2|3|AUX 3|0|ROOT 4|3|XJCT 5|4|OBJ 6|5...,it's 8065_9125 shocking 9125_9425 looking ...,ADULT,../data/chas/MICASE/interviews/int425jg002.cha,False,MICASE,MICASE-int425jg002-interviews,NaN
2,20,3,S1,very shocking .,"[11025, 12065]","[{'categoria': 'adv', 'lexema': 'very', 'extra...",1|2|JCT 2|0|ROOT 3|2|PUNCT,very 11025_11525 shocking 11525_12065 .,ADULT,../data/chas/MICASE/interviews/int425jg002.cha,False,MICASE,MICASE-int425jg002-interviews,NaN
3,24,4,S2,it's really normal .,"[12385, 13245]","[{'categoria': 'pro:per', 'lexema': 'it', 'ext...",1|2|SUBJ 2|0|ROOT 3|4|JCT 4|2|PRED 5|2|PUNCT,it's 12385_12505 really 12505_12765 normal...,ADULT,../data/chas/MICASE/interviews/int425jg002.cha,False,MICASE,MICASE-int425jg002-interviews,NaN
4,28,5,S2,it's what speech looks like .,"[14165, 15405]","[{'categoria': 'pro:per', 'lexema': 'it', 'ext...",1|2|SUBJ 2|0|ROOT 3|4|DET 4|2|PRED 5|4|ENUM 6|...,it's 14165_14245 what 14245_14385 speech ...,ADULT,../data/chas/MICASE/interviews/int425jg002.cha,False,MICASE,MICASE-int425jg002-interviews,NaN


In [11]:
data['conversation_id'].nunique()

152

### Correcting utterances/removing CLAN specific formatting.

In [12]:
def corrected_text(text, contraction_replacement_nonce='CCOONNTTRRAACCTTIIOONN'):
    res = re.sub(r'\(\(.*?\)\)', ' ', str(text))
    # res = re.sub(r'\[.*?\]', ' ', res)
    
    # find contractions and preserve them . . .
    contractions = list(re.findall(r"\w+'\w+", res))
    for contraction in set(contractions):
        replacement = contraction.replace("'", contraction_replacement_nonce)
        res = res.replace(contraction, replacement)
    res = re.sub(r"(⌋|⌊|⌉|⌈)", '', res)
    res = res.replace(':', '')
    
    # remove numbers in parentheses (times???)
    res = re.sub(r'\(\d\.\d\)', ' ', res)
    
    # remove all other special characters.
    res = re.sub(r'[^\w\s]', ' ', res)
    
    res = re.sub(r'\s+', ' ', res).replace('[', ' ').replace(']', ' ').replace(contraction_replacement_nonce, "'")
    return res.strip()

In [13]:
data['raw_text'] = data['text'].values
data['text'] = [corrected_text(text) for text in tqdm(data['raw_text'].values)]

  0%|          | 0/151542 [00:00<?, ?it/s]

In [14]:
data[['corpus', 'raw_text', 'text']].head(n=6)

,corpus,raw_text,text
0,MICASE,"+, my god I don't understand a word .",my god I don't understand a word
1,MICASE,&=laughs it's shocking looking isn't it ?,laughs it's shocking looking isn't it
2,MICASE,very shocking .,very shocking
3,MICASE,it's really normal .,it's really normal
4,MICASE,it's what speech looks like .,it's what speech looks like
5,MICASE,when you (.) take down everything .,when you take down everything


## Create juxtaposed corpus: (x,y) pairs

In [15]:
min_distance = 0
max_turns_apart = 200
save_at = 25

In [16]:
import warnings; warnings.filterwarnings("ignore")

corpus = []
for (count_,cid) in enumerate(tqdm(data['conversation_id'].unique())):

    sub = data.loc[data['conversation_id'].isin([cid])]
    sub_index = sub.index.values
    
    for i in sub_index:
        if i != sub_index[-1]:
            
            # speaker vs. other
            next_line_no = ( (sub_index > i) & (~sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][min_distance:(max_turns_apart+1)]
            
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]
            
            # speaker vs. self 
            next_line_no = ( (sub_index > i) & (sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][min_distance:(max_turns_apart+1)]
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]

    ### saving data iteratively via a series of checkpoints
    if ((count_+1) % save_at) == 0:
        corpus = pd.DataFrame(corpus)

        # rename columns to be appropriate
        rename_columns = dict()
        for col in corpus.columns:
            if col == 'text':
                rename_columns[col] = 'x_utterance'
            elif col == 'next_text':
                rename_columns[col] = 'y_utterance'
            elif 'next_' in col:
                rename_columns[col]  = col.replace('next', 'y')
            else:
                rename_columns[col] = col

        corpus = corpus.rename(columns=rename_columns)
        corpus = corpus.rename(columns={'speaker':'x_speaker','utterance_no': 'x_turn_id', 'y_utterance_no': 'y_turn_id'})

        # identify x and y unambiguously speaker by corpus and convo id
        corpus['x_speaker'] = corpus['corpus'].astype(str) + '-' + corpus['conversation_id'].astype(str) + '-' + corpus['x_speaker'].astype(str)

        corpus['y_speaker'] = corpus['corpus'].astype(str) + '-' + corpus['conversation_id'].astype(str) + '-' + corpus['y_speaker'].astype(str)

        # create indicator for self or not
        corpus['self'] = corpus['x_speaker'] == corpus['y_speaker']

        # sort and reindex
        corpus = corpus.sort_values(by=['corpus', 'conversation_id', 'x_turn_id', 'self', 'y_turn_id'])
        corpus.index = range(len(corpus))

        # save to parquet
        corpus[keep_columns].to_parquet(
            outputs_path.replace('.parquet', '-'+str(count_)+'.parquet'),
            engine='fastparquet',
            compression='snappy'
        )

        corpus = []

  0%|          | 0/152 [00:00<?, ?it/s]

In [17]:
if len(corpus):
    corpus = pd.DataFrame(corpus)

    # rename columns to be appropriate
    rename_columns = dict()
    for col in corpus.columns:
        if col == 'text':
            rename_columns[col] = 'x_utterance'
        elif col == 'next_text':
            rename_columns[col] = 'y_utterance'
        elif 'next_' in col:
            rename_columns[col]  = col.replace('next', 'y')
        else:
            rename_columns[col] = col
    corpus = corpus.rename(columns=rename_columns)
    corpus = corpus.rename(columns={'speaker':'x_speaker','utterance_no': 'x_turn_id', 'y_utterance_no': 'y_turn_id'})

    # identify x and y unambiguously speaker by corpus and convo id
    corpus['x_speaker'] = corpus['corpus'].astype(str) + '-' + corpus['conversation_id'].astype(str) + '-' + corpus['x_speaker'].astype(str)

    corpus['y_speaker'] = corpus['corpus'].astype(str) + '-' + corpus['conversation_id'].astype(str) + '-' + corpus['y_speaker'].astype(str)

    # create indicator for self or not
    corpus['self'] = corpus['x_speaker'] == corpus['y_speaker']

    # sort and reindex
    corpus = corpus.sort_values(by=['corpus', 'conversation_id', 'x_turn_id', 'self', 'y_turn_id'])
    corpus.index = range(len(corpus))

    # save to parquet
    corpus.to_parquet(
        outputs_path.replace('.parquet', '-'+str(count_)+'.parquet'),
        engine='fastparquet',
        compression='snappy'
    )